# Домашняя работа 1 — доли исходов HMDA

## Подготовка

In [8]:
from pathlib import Path
import sys
import pandas as pd

print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("Рабочая папка:", Path.cwd())
DATA = Path("../data")
assert DATA.is_dir(), "Рядом с папкой ноутбука должна находиться data/"

Python: C:\Users\Михаил Николаевич\AppData\Local\Programs\Python\Python313\python.exe
pandas: 3.1.0
Рабочая папка: F:\downolands\Курс по дата анализу\data-analysis-course-main\lessons\01-introduction\work


## Основной набор decisions.csv

In [2]:
home_decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
home_outcomes = home_decisions["action_taken"]
home_n = len(home_decisions)
assert home_n > 0
assert home_outcomes.isna().sum() == 0
assert home_outcomes.isin([1, 2, 3]).sum() == home_n
print(home_decisions.shape)
home_outcomes.value_counts(dropna=False)

(6830, 7)


action_taken
1    5956
3     691
2     183
Name: count, dtype: int64

## H1. Доля отказов

In [3]:

# Количество отказов
home_denied = (home_outcomes == 3).sum()

# Доля отказов среди всех записей
home_denial_rate = home_denied / home_n

print("Условие:", "home_outcomes == 3")
print("Числитель — количество отказов:", home_denied)
print("Знаменатель — общее количество записей:", home_n)
print("Доля отказов, %:", round(home_denial_rate * 100, 2))

# Количество одобрений с кодами 1 и 2
home_approved = home_outcomes.isin([1, 2]).sum()

print("Количество одобрений с кодами 1 и 2:", home_approved)

# Проверки
assert home_denied + home_approved == home_n
assert 0 <= home_denial_rate <= 1

print("Обе проверки пройдены.")

Условие: home_outcomes == 3
Числитель — количество отказов: 691
Знаменатель — общее количество записей: 6830
Доля отказов, %: 10.12
Количество одобрений с кодами 1 и 2: 6139
Обе проверки пройдены.


### Отчёт H1. Доля отказов

Доля-отказов-это отношение количества заявок с кодом action_taken = 3 к общему количеству записей в наборе home_decisions.

Количество отказов обозначается home_denied, а доля отказов-home_denial_rate. Для расчёта количество отказов делится на общее количество записей home_n.

В наборе данных всего 6830 записей, из которых 691 заявка получила отказ. Доля отказов составляет примерно 10,1%.

Таким образом, примерно каждая десятая запись в рассматриваемом наборе соответствует заявке, по которой был получен отказ. Проверка показала, что сумма отказов и одобрений с кодами 1 и 2 равна общему количеству записей, а доля отказов находится в диапазоне от 0 до 1.

## H2. Одобрение без принятия предложения

In [4]:
# Одобрения без принятия предложения
home_not_accepted = (home_outcomes == 2).sum()
home_not_accepted_rate = home_not_accepted / home_n

print("Условие:", "home_outcomes == 2")
print("Количество:", home_not_accepted)
print("Знаменатель:", home_n)
print("Доля, %:", round(home_not_accepted_rate * 100, 2))

# Количество выдач
home_issued = (home_outcomes == 1).sum()
home_issued_rate = home_issued / home_n

print("Количество выдач:", home_issued)
print("Доля выдач, %:", round(home_issued_rate * 100, 2))

# Проверяем связь исходов
assert home_approved - home_issued == home_not_accepted

# Разница долей одобрений и выдач
approval_rate = home_approved / home_n
difference_pp = (approval_rate - home_issued_rate) * 100

print("Разница долей, п. п.:", round(difference_pp, 2))
print("Проверка пройдена.")

Условие: home_outcomes == 2
Количество: 183
Знаменатель: 6830
Доля, %: 2.68
Количество выдач: 5956
Доля выдач, %: 87.2
Разница долей, п. п.: 2.68
Проверка пройдена.


### Отчёт H2. Одобрение без принятия предложения

Доля одобрений без принятия предложения-это отношение количества записей с кодом action_taken = 2 к общему количеству записей home_n.

В наборе данных 183 записи имеют код 2. При общем количестве 6830 записей их доля составляет примерно 2,68%.

Количество одобрений включает заявки с кодами 1 и 2, а количество выдач соответствует коду 1. Поэтому разность между количеством одобрений и количеством выдач равна количеству записей с кодом 2:

home_approved-home_issued == home_not_accepted.

Та же зависимость выполняется для долей, поскольку все они рассчитываются относительно одного знаменателя home_n. Разность доли одобрений и доли выдач составляет примерно 2,68 процентного пункта. Это означает, что такая доля всех записей относится к одобренным заявкам, по которым предложение не было принято.

## H3. Объяснение инструментов

### 1. Series и DataFrame
Series-это одномерная структура pandas, похожая на один столбец таблицы. Например, home_outcomes содержит исходы заявок.
DataFrame-это таблица, состоящая из строк и столбцов. Например, home_decisions содержит данные о заявках в табличном виде.
### 2. nrows и head()
nrows-параметр функции pd.read_csv(), который ограничивает количество строк, загружаемых из файла. Например, pd.read_csv("file.csv", nrows=10) загружает не более 10 строк.
head()-метод, который показывает первые строки уже загруженной таблицы. Например, home_decisions.head() показывает первые пять строк по умолчанию.
### 3. Метод и атрибут
Метод выполняет действие и вызывается со скобками. Например, home_decisions.head() показывает первые строки таблицы.
Атрибут предоставляет информацию об объекте и обычно вызывается без скобок. Например, home_decisions.shape возвращает количество строк и столбцов.
### 4. Работа с документацией
Я выбрал метод head() из библиотеки pandas и изучил его официальную документацию.
Из документации я уточнил параметр n: он определяет количество строк, которые нужно вывести. По умолчанию n=5, поэтому вызов home_decisions.head() показывает первые пять строк. Если указать home_decisions.head(3), будут показаны первые три строки

## Дополнительная часть

## Исходный снимок для H+1

In [5]:
raw = pd.read_csv(
    DATA / "hmda_2023_dc_raw.csv",
    dtype={"lei": "string", "county_code": "string"},
)
raw_outcomes = raw["action_taken"]
raw_n = len(raw)
assert raw_n > 0
print(raw.shape)
raw_outcomes.value_counts(dropna=False)

(8575, 99)


action_taken
1    5956
4    1585
3     691
2     183
5     160
Name: count, dtype: int64

## H+1. Доли выдач в двух наборах

In [6]:

# Исходный набор raw
raw_issued = (raw_outcomes == 1).sum()
raw_issued_rate = raw_issued / raw_n

print("Исходный набор raw")
print("Условие: raw_outcomes == 1")
print("Числитель:", raw_issued)
print("Знаменатель:", raw_n)
print("Доля выдач, %:", round(raw_issued_rate * 100, 2))

# Подготовленный набор home_decisions
home_issued = (home_outcomes == 1).sum()
home_issued_rate = home_issued / home_n

print("\nПодготовленный набор home_decisions")
print("Условие: home_outcomes == 1")
print("Числитель:", home_issued)
print("Знаменатель:", home_n)
print("Доля выдач, %:", round(home_issued_rate * 100, 2))

# Проверка совпадения количества выдач
assert raw_issued == home_issued

print("\nКоличество выдач в обоих наборах совпадает.")


Исходный набор raw
Условие: raw_outcomes == 1
Числитель: 5956
Знаменатель: 8575
Доля выдач, %: 69.46

Подготовленный набор home_decisions
Условие: home_outcomes == 1
Числитель: 5956
Знаменатель: 6830
Доля выдач, %: 87.2

Количество выдач в обоих наборах совпадает.


## Текстовые коды для H+2

In [7]:
text_example = pd.read_csv(
    DATA / "reading_example.csv", dtype={"action_taken": "string"}
)
text_example["action_taken"]

0    1
1    1
2    2
3    2
4    3
5    3
Name: action_taken, dtype: string

## H+2. Сравнение текста с числом и текстом